In [28]:
import socket
#import tensorflow as tensor
print(socket.gethostname())

fc30442


In [29]:
import numpy as np
import torch
print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")
#print(f"NumPy C-API: {hex(np.lib.NumpyVersion(np.__version__)._version)}")

NumPy:  1.25.2
PyTorch: 2.0.1


In [30]:
import torch
import numpy as np
import scipy.linalg as linalg
import matplotlib.pyplot as plt
print("Hello")

Hello


In [31]:
x = torch.Tensor(5, 3)
print(x)
y = torch.rand(5, 3)
print(y)
# let us run the following only if CUDA is available
if torch.cuda.is_available():
    x = x.cuda()
    y = y.cuda()
    print("There is cuba")
    print(x + y)
print("Reach end")

tensor([[0.0000e+00, 7.4829e-42, 7.7052e+31],
        [7.2148e+22, 1.5766e-19, 1.0256e-08],
        [2.6252e-06, 3.0613e+12, 4.3357e-08],
        [1.0742e-05, 2.6554e-06, 1.2961e+16],
        [2.1707e-18, 7.0952e+22, 1.7748e+28]])
tensor([[0.3211, 0.6614, 0.1385],
        [0.6130, 0.9058, 0.5723],
        [0.6801, 0.5450, 0.3730],
        [0.9310, 0.9007, 0.0932],
        [0.9516, 0.0123, 0.0289]])
Reach end


In [32]:
def gaussian3d(x,mean,covar = np.array([[5,0,0],[0,5,0],[0,0,5]])):

    '''
    returns a 3d gaussian evaluated for an image and time location with a given covariance matrix and mean
    '''
    

    Ninv = np.linalg.pinv(covar)

    return np.exp(-np.diag((x-mean).T@Ninv@(x-mean))) ## there's probably a smarter way to do this
    #Normalized: 

    #return ((2*np.pi)**-1.5)*(np.linalg.det(covar)**-0.5)*np.exp(-0.5*np.diag((x-mean).T@Ninv@(x-mean)))


def genrandomfield(xlen,ylen,tlen,Ntrans,widths = None,noiseamp = 0.1):

    '''
    generates a field of white noise with a given number of overlaid gaussian 'transients'.
    Transients and noise returned as separate layers. 

    Arguements (in order):

    xlen: length of the first image dimension, in pixels
    ylen: same as above but for second dimension
    tlen: length of the image/noise fields to be generated, in time (arbitrary units)
    Ntrans: how many transients to simulate
    widths: a 3-tuple (A,B,C) of sigma values corresponding to the x, y and time widths of the 3D Gaussian transients.
    for example, widths = (1,2,10) will give transients that are narrow in the first image dimension, broader in the second,
    and have a characteristic timescale of about 10 time units.
    noiseamp: the root-mean-square level of (white) noise in the image.

    returns: (in order)
    combinedfield, noisefield+transient_field
    noisefield: the noise layer, with shape (xlen,ylen,tlen)
    transient_field: the transient layer, with shape (xlen,ylen,tlen)
    x0: the list of positions of each source in the first image axis
    y0: the list of positions of each source in the second image axis
    t0: the central times of each transient signal (time of peak brightness)
    '''

    x = np.arange(xlen) ## x and y arrays
    y = np.arange(ylen)

    xx,yy = np.meshgrid(x,y) ## create a grid of x,y positions to evaluate at

    xxf,yyf = xx.ravel(),yy.ravel() ## flatten for processing

    rng = np.random.default_rng()

    noisefield = noiseamp*rng.standard_normal(size = (xlen,ylen,tlen)) # generate the noise

    transient_field = np.zeros(noisefield.shape) ## array for the transient layer to be created in
    combined_field = np.zeros(noisefield.shape)

    x0 = [] ## these record the x, y, and time mean values for each transient signal
    y0 = [] ## these record the x, y, and time mean values for each transient signal
    t0 = [] ## these record the x, y, and time mean values for each transient signal

    #x_std = [] #record gaussian std on x
    #y_std = [] #record gaussian std on y
    #t_std = [] #record gaussian std on t

    posvecs = np.zeros((3,len(xxf)))
    posvecs[0,:] = xxf
    posvecs[1,:] = yyf

    if widths:
        covar = np.array([[widths[0]**2,0,0],[0,widths[1]**2,0],[0,0,widths[2]**2]])
    
    for i in range(Ntrans):

        x0.append(np.random.randint(xlen)) ## record where this trans happened
        y0.append(np.random.randint(ylen))
        t0.append(np.random.randint(tlen))
        #x_std.append(np.random.randint(1, xlen))

        for t in range(tlen):

            posvecs[2,:] = t

            if widths:

                trans = gaussian3d(posvecs, mean = np.array([[x0[-1],y0[-1],t0[-1]]]).T, covar = covar).reshape((xlen,ylen))
            else:

                trans = gaussian3d(posvecs, mean = np.array([[x0[-1],y0[-1],t0[-1]]]).T).reshape((xlen,ylen))

            transient_field[:,:,t] += trans
            combined_field[:,:,t] = transient_field[:,:,t] + noisefield[:,:,t]

    median = np.vstack((x0[-1], y0[-1], t0[-1], widths[0], widths[1], widths[2]))
    return combined_field, noisefield,transient_field,median

In [33]:
def genrandomfields(data_size = 1, Ntrans = 5, xlen=50, ylen=50, tlen=100):
    widths = (5, 5, 10)
    combined_fields = []
    medians = []
    for datum in range(data_size):
        combined, noise, transient, median = genrandomfield(xlen,ylen,tlen,Ntrans,widths)
        combined_fields.append(combined)
        medians.append(median)
    combined_fields = np.array(combined_fields)
    medians = np.array(medians)
    medians = medians[:,:,0]
    return combined_fields, medians

In [34]:
training,n_0,t_0,training_median = genrandomfield(50,50,100,1, widths = (5,5,10))#, noiseamp=1e-10)
print(training_median.shape)

(6, 1)


In [35]:
# generates and saves animation frames for the above generated field

for frame in range(t_0.shape[-1]):
    plt.figure(figsize = (5,5))
    plt.imshow(training[:,:,frame],vmax = 1,vmin = 0)
    plt.xticks([])
    plt.yticks([])
    #plt.show()
    #plt.savefig('example_animation/frame_%d.png' %frame)
    plt.close()


In [36]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
#from torchvision import datasets, transforms
print("Os imported")
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
#device = "mps"
print(f"Using {device} device")

Os imported
Using cpu device


In [37]:
def normalize_data(data):
    mean = data.mean()
    std = data.std()
    return (data - mean) / std, mean, std

In [38]:
def convert_to_torch_data(data):
    data = torch.tensor(data, dtype=torch.float32)
    data = data.squeeze(0)
    #data = data.permute(0, 3, 1, 2)
    data = data.float().to(device)
    return normalize_data(data)

In [39]:
X_test, y_test = genrandomfields(10, 1, 50, 50, 100)
X_test,_,_ = convert_to_torch_data(X_test)
y_test,_,_ = convert_to_torch_data(y_test)
print("Testing set gained")
print(X_test.shape)
print(y_test.shape)

Testing set gained
torch.Size([10, 50, 50, 100])
torch.Size([10, 6])


In [40]:
X_train, y_train = genrandomfields(1000, 1, 50, 50, 100)
X_train,_,_ = convert_to_torch_data(X_train)
y_train,y_train_mean,y_train_std = convert_to_torch_data(y_train)
print("Training set gained")
print(X_train.shape)
print(y_train.shape)

KeyboardInterrupt: 

In [ ]:
"""
#X = normalize_data(X)
#y = normalize_data(y)
X_train = convert_to_torch_data(X)
X_train = X_train.permute(0, 3, 1, 2)
y_train = convert_to_torch_data(y)
print(X_train.shape)
X_train.to(device)
y_train.to(device)
y_train = y_train[:, :, 0]
X, y = genrandomfields(100, 1, 50, 50, 100)
#X = normalize_data(X)
#y = normalize_data(y)
X_test = convert_to_torch_data(X)
X_test = X_test.permute(0, 3, 1, 2)
y_test = convert_to_torch_data(y)
X_test.to(device)
y_test.to(device)
y_test = y_test[:, :, 0]
y_train.to(torch.long)
y_test.to(torch.long)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(y_test.dtype)
"""

In [ ]:
'''
X_train, _, _ = normalize_data(X_train)
y_train, y_train_mean, y_train_std  = normalize_data(y_train)
X_test, _, _  = normalize_data(X_test)
y_test, _, _  = normalize_data(y_test)
'''

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test,  y_test)

batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(50*50*100, 2048),
            nn.ReLU(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        #x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
model = NeuralNetwork().to(device)
print(model)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0
    for X, y in dataloader:
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def test_loop(dataloader, model, loss_fn):
    model.eval()
    num_batches = len(dataloader)
    test_loss = 0
    all_preds = []

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            all_preds.append(pred)

    test_loss /= num_batches
    all_preds = torch.cat(all_preds, dim=0)
    return all_preds, test_loss  # return both

In [ ]:
'''
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        return loss

def test_loop(dataloader, model, loss_fn):
    model.eval()
    num_batches = len(dataloader)
    test_loss = 0
    all_preds = []

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            all_preds.append(pred)

    test_loss /= num_batches
    print(f"Avg loss: {test_loss:>8f}")
    
    all_preds = torch.cat(all_preds, dim=0)
    return all_preds

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    all_preds = []

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            #print("The pred shape is: ", pred.shape)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
            all_preds.append(pred)
            #print("The y shape is:", y.shape)

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    all_preds = torch.cat(all_preds, dim=0)  
    return all_preds
'''

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

learning_rate = 1e-3
batch_size = 64
epochs = 500
train_losses = []
test_losses = []

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5, verbose=True)

for t in range(epochs):
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer)
    preds, test_loss = test_loop(test_dataloader, model, loss_fn)
    scheduler.step(test_loss)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    clear_output(wait=True)
    plt.plot(train_losses, label='Train')
    plt.plot(test_losses, label='Test')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Epoch {t+1}, Train: {train_loss:.4f}, Test: {test_loss:.4f}')
    plt.legend()
    plt.show()

print("Done!")

In [1]:
X, y = genrandomfields(10, 1, 50, 50, 100)
X_eval = convert_to_torch_data(X)
X_eval = X_eval.permute(0, 3, 1, 2)
y_eval = convert_to_torch_data(y)
X_eval.to(device)
y_eval.to(device)
y_eval = y_eval[:, :, 0]
y_eval.to(torch.long)
X_eval, _, _ = normalize_data(X_eval)
#y_eval, _, _ = normalize_data(y_eval)

new_dataset = TensorDataset(X_eval, y_eval)
new_dataloader = DataLoader(new_dataset, batch_size=batch_size)

preds, loss = test_loop(new_dataloader, model, loss_fn)

#true_values = torch.cat([y for _, y in new_dataloader], dim=0)

preds = y_train_mean + preds * y_train_std
print(y_eval)
print(preds)

NameError: name 'genrandomfields' is not defined

In [ ]:
'''
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")
'''

In [ ]:
#custom dataset
'''
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) 

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
'''

In [ ]:
'''
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")
'''